# Notebook 03: Text Splitting

In this notebook, we will learn how to **split large documents into smaller chunks**.

## What You Will Learn

- Why we need to split documents
- How LangChain's RecursiveCharacterTextSplitter works
- What chunk size and overlap mean
- How to preserve metadata during splitting

## Why Split Documents?

Language models have a **context window** - a maximum number of tokens they can process at once. For example:

| Model | Context Window |
|-------|---------------|
| Mistral | ~4096 tokens |
| GPT-4 | ~8192 tokens |
| Claude | ~200,000 tokens |

If our PDF has 50 pages, we can't send all of it to the model at once. We need to:

1. Split the text into manageable **chunks**
2. Send only the most relevant chunks to the model
3. This is the **"Retrieval"** part of RAG

## What is chunk_size and chunk_overlap?

- **chunk_size**: The maximum number of characters in each chunk (e.g., 500)
- **chunk_overlap**: How many characters overlap between consecutive chunks (e.g., 100)

```
Chunk 1: [___________________________][overlap]
Chunk 2:                       [overlap][___________________________]
```

The overlap ensures that no important information is cut off at chunk boundaries.

## Step 1: Import Required Libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PyPDFLoader: loads PDF files
# RecursiveCharacterTextSplitter: splits text into chunks

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_3704\2751110023.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Step 2: Load the PDF

In [2]:
# Load the PDF
loader = PyPDFLoader("../data/sample.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print(f"Total characters: {sum(len(p.page_content) for p in pages)}")

Loaded 4 pages
Total characters: 7712


## Step 3: Create a Text Splitter

In [3]:
# Create a RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Maximum characters per chunk
    chunk_overlap=100,     # Overlapping characters between chunks
    length_function=len,   # How to measure chunk length
    separators=["\n\n", "\n", ". ", " ", ""]  # Where to split (in order of preference)
)

print("Text splitter created successfully!")
print(f"Chunk size: {text_splitter._chunk_size}")
print(f"Chunk overlap: {text_splitter._chunk_overlap}")

Text splitter created successfully!
Chunk size: 500
Chunk overlap: 100


## Step 4: Split the Documents

The splitter tries to break text at paragraph breaks first, then at sentence boundaries, then at word boundaries, and finally at character level.

In [4]:
# Split all pages into chunks
chunks = text_splitter.split_documents(pages)

print(f"Total chunks created: {len(chunks)}")
print(f"\nFirst chunk:")
print(f"Characters: {len(chunks[0].page_content)}")
print(f"Content: {chunks[0].page_content}")
print(f"\nMetadata: {chunks[0].metadata}")

Total chunks created: 20

First chunk:
Characters: 468
Content: Introduction to Artificial Intelligence
Chapter 1: What is Artificial Intelligence?
Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of
performing tasks that typically require human intelligence. These tasks include understanding natural
language, recognizing patterns, solving problems, and making decisions.
The concept of AI was first introduced in 1956 at the Dartmouth Conference, where a group of researchers

Metadata: {'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260729184321', 'source': '../data/sample.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


## Step 5: Analyze Chunk Sizes

In [5]:
# Let's look at the distribution of chunk sizes
sizes = [len(chunk.page_content) for chunk in chunks]

print(f"Minimum chunk size: {min(sizes)} characters")
print(f"Maximum chunk size: {max(sizes)} characters")
print(f"Average chunk size: {sum(sizes) / len(sizes):.1f} characters")

# Show first 5 chunks
print("\n--- First 5 Chunks ---")
for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i+1} ({len(chunk.page_content)} chars):")
    print(chunk.page_content[:150] + "..." if len(chunk.page_content) > 150 else chunk.page_content)

Minimum chunk size: 187 characters
Maximum chunk size: 491 characters
Average chunk size: 416.8 characters

--- First 5 Chunks ---

Chunk 1 (468 chars):
Introduction to Artificial Intelligence
Chapter 1: What is Artificial Intelligence?
Artificial Intelligence (AI) is a branch of computer science that ...

Chunk 2 (422 chars):
gathered to explore the possibility of creating machines that could think. Since then, AI has evolved
significantly and is now used in many industries...

Chunk 3 (480 chars):
performing any intellectual task that a human can do. Super AI refers to a hypothetical AI that surpasses
human intelligence in every aspect.
Chapter ...

Chunk 4 (461 chars):
statistical techniques to find patterns in data.
There are three main types of Machine Learning:
1. Supervised Learning: The algorithm learns from lab...

Chunk 5 (259 chars):
structures. Examples include customer segmentation and anomaly detection.
3. Reinforcement Learning: The algorithm learns by interacting with a

## Step 6: Experiment with Different Parameters

Let's see how different chunk sizes affect the number of chunks.

In [6]:
# Try different chunk sizes
chunk_sizes = [200, 500, 1000, 2000]

for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=size // 5  # 20% overlap
    )
    result = splitter.split_documents(pages)
    print(f"Chunk size {size}: {len(result)} chunks (avg {sum(len(c.page_content) for c in result) // len(result)} chars)")

Chunk size 200: 52 chunks (avg 151 chars)
Chunk size 500: 20 chunks (avg 416 chars)
Chunk size 1000: 12 chunks (avg 749 chars)
Chunk size 2000: 5 chunks (avg 1617 chars)


## Key Takeaways

1. **RecursiveCharacterTextSplitter** is the most commonly used splitter
2. It tries to split at natural boundaries (paragraphs, sentences, words)
3. **chunk_size** controls how big each chunk is
4. **chunk_overlap** ensures context isn't lost at boundaries
5. Metadata (like page number) is preserved in each chunk

## Choosing Chunk Size

- Too small: chunks may lack context
- Too large: may exceed model's context window
- **500-1000 characters** is a good starting point for most documents

## Next Steps

Proceed to **Notebook 04: Create Embeddings** to learn how to convert text chunks into vectors.